# Retail product detection: YOLOv8n vs YOLOv8s

Dataset: **Sell Products** (O2O Minimart, Roboflow Universe, CC BY 4.0). 110 product classes.

Notebook นี้รันได้ทั้ง Google Colab และ Kaggle (เปิด GPU ก่อน) ทำงานตามลำดับนี้

| ขั้น | ส่วน | ผลลัพธ์ |
|---|---|---|
| 0 | Setup | clone repo, ติดตั้งแพ็กเกจ |
| 1 | ดึงข้อมูล (ข้อ 5.1) | `datasets/sell-products/` |
| 2 | EDA (ข้อ 5.2) | `results/eda_*` |
| 3 | เทรน YOLOv8n / YOLOv8s (ข้อ 5.3–5.5) | `runs/detect/<model>/weights/best.pt` |
| 4 | ประเมินบน test set (ข้อ 5.6, 6) | `results/*.csv`, `results/*.png` |
| 5 | อัปเดต README (ข้อ 7) | ตาราง + ข้อค้นพบใน `README.md` |
| 6 | commit + push | เฉพาะ `results/` และ `README.md` |

**รันซ้ำเฉพาะส่วนประเมิน:** ถ้าเคยเทรนไว้แล้วและเก็บ `runs/` ใน Google Drive (ไฟล์ `DONE` อยู่ในโฟลเดอร์ของแต่ละโมเดล)
ขั้นที่ 3 จะข้ามการเทรนเอง กด Run all ได้เลย ใช้เวลาแค่ดาวน์โหลดข้อมูล + ประเมิน (~10–15 นาทีบน T4)

**ก่อนรัน:** เพิ่ม secret ชื่อ `ROBOFLOW_API_KEY` (Colab: ไอคอนกุญแจทางซ้าย → เปิด *Notebook access*;
Kaggle: Add-ons → Secrets) ห้ามวาง key ลงใน notebook

## 0. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU - Runtime > Change runtime type > T4 GPU"

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/ToonRz/retail-shelf-detection.git"
BRANCH = "main"            # branch ที่มีโค้ดนี้อยู่

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()
IN_KAGGLE = Path("/kaggle/working").exists()
BASE = Path("/kaggle/working") if IN_KAGGLE else Path("/content") if IN_COLAB else None

if BASE is not None:                       # cloud notebook -> clone repo
    ROOT = BASE / "retail-shelf-detection"
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(ROOT)], check=True)
    else:  # re-run in the same session: get the latest code. Generated README/results are
        # discarded here (they are rebuilt below); download results.zip first if you need them.
        subprocess.run(["git", "-C", str(ROOT), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(ROOT), "reset", "--hard", f"origin/{BRANCH}"], check=True)
        print("updated to latest", BRANCH, "- if src/ changed, Runtime > Restart session, then Run all")
else:                                      # local: notebook lives in notebooks/
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("project root:", ROOT)

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
# ---------------- experiment config (เหมือนกันทั้งสองโมเดล เพื่อให้เทียบแฟร์) ----------------
DATASET_VERSION = 3        # ดูในแท็บ Dataset บน Roboflow
MODELS = ["yolov8n", "yolov8s"]
MODELS_TO_TRAIN = MODELS   # ถ้าแยกรัน Colab/Kaggle: ["yolov8n"] ที่หนึ่ง, ["yolov8s"] อีกที่
EPOCHS, IMGSZ, BATCH, PATIENCE, SEED = 25, 640, 16, 5, 0

# เก็บ runs/ ไว้ใน Google Drive กัน Colab หลุดแล้ว weights หาย (แนะนำ)
SAVE_TO_DRIVE = IN_COLAB and not IN_KAGGLE
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUNS_DIR = Path("/content/drive/MyDrive/retail-shelf-detection/runs/detect")
else:
    RUNS_DIR = ROOT / "runs" / "detect"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)
print("runs ->", RUNS_DIR)

## 1. ดึงข้อมูล (ข้อ 5.1)
ใช้ Roboflow API (export format `yolov8`) และใช้ split train / valid / test ตามที่ Roboflow แบ่งไว้
`fix_data_yaml` เขียน `data_abs.yaml` ใหม่ด้วย absolute path เพราะ `data.yaml` ของ Roboflow ใช้ `../train/images` ซึ่งมักจะ path ผิด

In [ ]:
from data import download_dataset, fix_data_yaml, load_names

DS_DIR = download_dataset(DATASET_VERSION, ROOT / "datasets" / "sell-products")
DATA_YAML = fix_data_yaml(DS_DIR)
NAMES = load_names(DATA_YAML)
print(DATA_YAML.read_text()[:400], "...")
print(f"{len(NAMES)} classes, e.g.", NAMES[:8])

## 2. EDA (ข้อ 5.2)
นับจำนวนรูป/กล่องต่อ split, การกระจายของคลาส, จำนวนกล่องต่อรูป และขนาดกล่องเทียบกับรูป
(ใช้ยืนยันว่ารูปส่วนใหญ่เป็นสินค้าชิ้นเดียวถ่ายใกล้ ๆ ไม่ใช่ชั้นวางแน่น ๆ)

In [ ]:
import json, pandas as pd
from IPython.display import Image as ShowImage, display
from data import run_eda

eda = run_eda(DS_DIR, NAMES, RESULTS)
json.dump(eda, open(RESULTS / "eda_summary.json", "w"), indent=2, ensure_ascii=False)
display(pd.read_csv(RESULTS / "eda_splits.csv"))
print("imbalance (max/min, train):", eda["train_imbalance_ratio"])
print("rarest:", eda["rarest_classes"])
print("image sizes:", eda["image_sizes"], "checked:", eda["images_checked_for_size"])
print("near-duplicate check (test vs train burst frames):", eda.get("near_duplicates"))
display(ShowImage(RESULTS / "eda_class_distribution.png"))
display(ShowImage(RESULTS / "eda_boxes.png"))

## 3. เทรน (ข้อ 5.3–5.5)
* **Transfer learning**: เริ่มจาก weights ที่ pretrain บน COCO (`yolov8n.pt`, `yolov8s.pt`)
* **Preprocessing / augmentation**: letterbox resize 640 + augmentation ในตัวของ YOLO
  (mosaic, horizontal flip 0.5, HSV jitter, scale/translate) ปิด mosaic ใน 10 epoch สุดท้าย
* config เดียวกันทั้งสองโมเดล: `epochs=25, imgsz=640, batch=16, patience=5, seed=0`, optimizer auto (SGD/AdamW)
* ถ้า Colab หลุด ให้กด **Run all ใหม่** (VM ถูกล้าง ต้องโหลดข้อมูลใหม่) ขั้นนี้จะ **resume** จาก `last.pt` ใน Drive
  และข้ามโมเดลที่เทรนเสร็จแล้ว (มีไฟล์ `DONE`)

> ⏱ บน T4 ประมาณ v8n ~2–3 นาที/epoch, v8s ~3–5 นาที/epoch (ขึ้นกับจำนวนรูป train)
> ถ้าเวลาไม่พอให้ลด `EPOCHS` เหลือ 15 (`patience` จะหยุดก่อนอยู่แล้วถ้า mAP ไม่ขึ้น)

In [ ]:
from ultralytics import YOLO

def train(name):
    run = RUNS_DIR / name
    last, done = run / "weights" / "last.pt", run / "DONE"
    if done.exists():
        print(f"[{name}] already trained - skip"); return
    if last.exists():
        import torch
        ck = torch.load(str(last), map_location="cpu", weights_only=False)
        if ck.get("epoch", -1) == -1 and (run / "weights" / "best.pt").exists():
            print(f"[{name}] last.pt is a finished checkpoint - marking done")
            done.write_text("ok"); return
        print(f"[{name}] resuming from {last}")
        YOLO(str(last)).train(resume=True)
    else:
        YOLO(f"{name}.pt").train(
            data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
            patience=PATIENCE, seed=SEED, deterministic=True,
            project=str(RUNS_DIR), name=name, exist_ok=True,
            workers=2, plots=True,
            # augmentation (ค่า default ของ Ultralytics เขียนไว้ให้เห็นชัด)
            mosaic=1.0, close_mosaic=10, fliplr=0.5,
            hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, scale=0.5, translate=0.1,
        )
    done.write_text("ok")

for m in MODELS_TO_TRAIN:
    train(m)

In [ ]:
# กราฟ loss / mAP ต่อ epoch ที่ Ultralytics สร้างให้
for m in MODELS:
    p = RUNS_DIR / m / "results.png"
    if p.exists():
        print(m); display(ShowImage(p, width=900))

## 4. ประเมินผลบน test set (ข้อ 5.6 และ ข้อ 6)
`evaluate_all` ทำทุกอย่างให้และเขียนลง `results/`:

* **Detection metrics** (`model.val(split="test")`): Precision, Recall, F1, mAP@0.5, mAP@0.5:0.95 และ per-class AP
  พร้อมก๊อป `PR_curve.png`, `F1_curve.png` และ confusion matrix ที่ conf 0.25 (`<model>_confusion_matrix_conf0.25.png`) มาไว้
* **Image-level metrics** (เพื่อรายงาน Accuracy / AUC ตามโจทย์):
  label จริงของรูป = คลาสที่มีกล่องมากที่สุดในรูป, คำทำนาย = คลาสของกล่องที่ conf สูงสุด (ต้อง ≥ 0.25 ไม่งั้นนับว่า *no detection* = ผิด),
  score ของแต่ละคลาส = conf สูงสุดของกล่องคลาสนั้น → sklearn accuracy, macro P/R/F1, ROC-AUC แบบ one-vs-rest
* **ความเร็ว**: median ms/รูป ที่ batch = 1 (เหมือนกล้องในร้านที่ส่งมาทีละภาพ)
* ความผิดพลาดระดับกล่อง (missed / wrong class / IoU ต่ำ / กล่องเกิน) + กริด `examples_box_errors_<model>.jpg`
* confusion matrix ที่ conf 0.25 (`confusion_matrix_conf0.25_<model>.csv`), คลาสที่ยากที่สุด, กราฟรายคลาส
* บันทึก optimizer/lr ที่ใช้จริง เวลาเทรน ชื่อ GPU และเวอร์ชันแพ็กเกจ (`metrics_<model>.json`, `run_info.json`)
* เก็บผลดิบ (`predictions_<model>.jsonl`, `scores_<model>.npy`) ไว้คำนวณซ้ำได้โดยไม่ต้องใช้ GPU

In [ ]:
from evaluate import evaluate_all

weights = {m: RUNS_DIR / m / "weights" / "best.pt" for m in MODELS}
missing = [m for m, w in weights.items() if not w.exists()]
assert not missing, f"no best.pt for {missing} - train first (or copy weights from Kaggle into {RUNS_DIR})"

table = evaluate_all(weights, DATA_YAML, out_dir=RESULTS, split="test", imgsz=IMGSZ,
                     project=RUNS_DIR, train_dirs={m: RUNS_DIR / m for m in MODELS})
table.T

In [ ]:
for f in ["comparison_bar.png", "speed_accuracy.png", "training_curves.png", "roc_curve.png",
          "per_class_scatter.png", "hardest_classes.png", "box_confusions.png",
          "examples_box_errors_yolov8n.jpg", "examples_box_errors_yolov8s.jpg",
          "examples_correct_yolov8s.jpg", "examples_wrong_yolov8s.jpg",
          "yolov8n_confusion_matrix_conf0.25.png", "yolov8s_confusion_matrix_conf0.25.png"]:
    if (RESULTS / f).exists():
        print(f); display(ShowImage(RESULTS / f, width=1000))
print(open(RESULTS / "disagreement.json").read())

## 5. อัปเดต README (ข้อ 7)
เติมตารางผล, ข้อค้นพบที่เป็นตัวเลข, top confusions และคลาสที่ยากที่สุด ลงในช่อง `AUTO` ของ README
(ส่วนที่เขียนวิเคราะห์ด้วยมือจะไม่ถูกแตะ) จากนั้นเปิดดู `results/examples_box_errors_<model>.jpg`,
`results/box_confusions_<model>.csv` (และ `box_confusions.png` ถ้ามีการสลับคลาส ≥ 3 คู่)
แล้วแก้ย่อหน้า *การวิเคราะห์* ใน README ให้ตรงกับผลจริง

In [ ]:
import make_report
make_report.main(RESULTS, ROOT / "README.md")
print((RESULTS / "comparison.md").read_text())

## 6. Commit + push
commit **เฉพาะ** `results/` กับ `README.md` (dataset, `runs/`, `*.pt`, `.env` ถูก `.gitignore` กันไว้แล้ว)

* **ทางเลือก A (ง่ายสุด):** ดาวน์โหลด `results.zip` จาก cell ล่าง แตกไฟล์ที่ root ของ repo (`unzip -o results.zip`)
  จะได้ `results/` และ `README.md` ใหม่ แล้ว commit/push เอง (หรือส่ง zip ให้ผู้ช่วยทำต่อ)
* **ทางเลือก B:** push จาก Colab โดยตรง ตั้ง `PUSH = True` ใน cell ล่างสุด และต้องมี secret `GITHUB_TOKEN`
  (fine-grained token, สิทธิ์ Contents: write เฉพาะ repo นี้)

In [ ]:
# ทางเลือก A: zip มี results/ และ README.md ที่เติมตัวเลขแล้ว -> แตกไฟล์ที่ root ของ repo ทับได้เลย
import zipfile
with zipfile.ZipFile(ROOT / "results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(RESULTS.rglob("*")):
        if f.is_file():
            z.write(f, f.relative_to(ROOT))
    z.write(ROOT / "README.md", "README.md")
print("results.zip:", round((ROOT / "results.zip").stat().st_size / 2**20, 1), "MB")
if IN_COLAB and not IN_KAGGLE:
    from google.colab import files
    files.download(str(ROOT / "results.zip"))

In [ ]:
# ทางเลือก B (ไม่บังคับ)
PUSH = False
GIT_NAME, GIT_EMAIL = "Your Name", "you@example.com"
if not PUSH:
    print("PUSH = False - nothing pushed (use results.zip, or set PUSH = True)")
else:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    remote = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
    def git(*a):
        r = subprocess.run(["git", *a], capture_output=True, text=True)
        print((r.stdout + r.stderr).replace(token, "***"))
    git("config", "user.name", GIT_NAME); git("config", "user.email", GIT_EMAIL)
    git("add", "results", "README.md")
    git("status", "--short")
    git("commit", "-m", "eval: metrics + plots on test set")
    git("push", remote, f"HEAD:{BRANCH}")